# **Project introduction**

# AI Knowledge Assistant using Structure-Aware RAG

## Objective
Build a document question-answering system for the Sony WH-1000XM6 user manual.

## Pipeline
1. PDF text extraction
2. Document cleaning
3. Chapter and section detection
4. Section-aware chunking
5. MiniLM embeddings
6. Chroma vector database
7. Semantic retrieval
8. Qwen answer generation

## Key improvement
Instead of splitting the PDF using a fixed character count alone,
the system detects document sections first and keeps metadata
such as chapter, section, and source PDF page numbers.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running without CUDA GPU")

CUDA available: False
Running without CUDA GPU


# **Import modules**

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path(
    "/content/drive/MyDrive/sony_rag_assistant"
)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)
print("Files:", [p.name for p in PROJECT_DIR.iterdir()])

Project directory: /content/drive/MyDrive/sony_rag_assistant
Files: ['structure_extractor.py', 'document_cleaner.py', 'config.py', 'generator.py', 'chunker.py', 'vector_store.py', 'rag_pipeline.py', 'retriever.py', 'README.md', 'document_parser.py', 'data', 'chroma_db', '__pycache__', '.gradio', 'scr', 'app.py', 'requirements.txt', 'demo.ipynb']


In [14]:
%pip install -q groq

In [ ]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass(
    "Enter your Groq API Key: "
)

print("GROQ_API_KEY configured:", bool(os.environ["GROQ_API_KEY"]))

In [31]:
import json
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/sony_rag_assistant_restructured"
)
notebook_path = project_dir / "demo.ipynb"

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

for i in [7, 11]:
    cell = notebook["cells"][i]

    print(f"\n--- Cell {i} ---")
    print("Code:")
    print("".join(cell.get("source", [])))

    for j, output in enumerate(cell.get("outputs", [])):
        output_text = json.dumps(output, ensure_ascii=False)
        suspicious = any(
            marker in output_text.lower()
            for marker in ["gsk_", "hf_", "api_key", "authorization: bearer"]
        )
        print(
            f"Output {j}: type={output.get('output_type')}, "
            f"contains_sensitive_marker={suspicious}"
        )


--- Cell 7 ---
Code:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass(
    "Enter your Groq API Key: "
)

print("GROQ_API_KEY configured:", bool(os.environ["GROQ_API_KEY"]))
Output 0: type=stream, contains_sensitive_marker=True

--- Cell 11 ---
Code:
from pathlib import Path
from IPython.display import display, Markdown

from config import DATA_DIR, TOP_K
from rag_pipeline import RAGPipeline

pipeline = RAGPipeline()

# Automatically find a PDF file in the data folder
pdf_files = list(DATA_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF file found in {DATA_DIR}. Please upload your PDF first."
    )

pdf_path = pdf_files[0]

print("PDF exists:", pdf_path.exists())
print("PDF path:", pdf_path)
Output 0: type=stream, contains_sensitive_marker=True
Output 1: type=display_data, contains_sensitive_marker=False
Output 2: type=display_data, contains_sensitive_marker=False
Output 3: type=display_data, contains_sensitive_marker=False
Out

# **Install dependencies**

In [16]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 65.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 32.5 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.2 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully 

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

from config import DATA_DIR, TOP_K
from rag_pipeline import RAGPipeline

pipeline = RAGPipeline()

# Automatically find a PDF file in the data folder
pdf_files = list(DATA_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF file found in {DATA_DIR}. Please upload your PDF first."
    )

pdf_path = pdf_files[0]

print("PDF exists:", pdf_path.exists())
print("PDF path:", pdf_path)

# **PDF parsing**

## Step 1: PDF Parsing

Extract text from each page and preserve its PDF page number.

In [ ]:
from document_parser import extract_pdf_pages

pages = extract_pdf_pages(str(pdf_path))

print("Total extracted pages:", len(pages))
print("\nFirst page preview:\n")
print(pages[0]["text"][:1500])

Total extracted pages: 146

First page preview:

Help Guide


Wireless Noise Canceling Stereo Headset
WH-1000XM6





Model: YY2984

Quick Start Guide

  Find your fit

Wearing the headset

 Download the app, and set up the headset

Many useful features are out there in your device.
Download the “Sony | Sound Connect” and get the most out of what you have.





  Let’s start to use


                Turning on the headset

                Press and hold the    (power) button for about 2 seconds or more until the indicator (blue) flashes.
              The headset enters pairing mode automatically when you turn on the headset for the first time after
                 purchase.





                                                          1


# **Cleaning**

In [ ]:
from document_cleaner import clean_pages

cleaned_pages = clean_pages(pages, remove_toc=False)

print("Pages after cleaning:", len(cleaned_pages))
print("\nFirst page preview:\n")
print(cleaned_pages[0]["text"][:1500])

Pages after cleaning: 143

First page preview:

Model: YY2984
Quick Start Guide
Find your fit
Wearing the headset
Download the app, and set up the headset
Many useful features are out there in your device.
Download the “Sony | Sound Connect” and get the most out of what you have.
Let’s start to use
Turning on the headset
Press and hold the (power) button for about 2 seconds or more until the indicator (blue) flashes.
The headset enters pairing mode automatically when you turn on the headset for the first time after
purchase.


# **Structure extraction**

In [ ]:
from structure_extractor import extract_document_structure

sections = extract_document_structure(cleaned_pages)

print("Sections detected:", len(sections))

for i, section in enumerate(sections[:10], start=1):
    print(f"\nSection {i}")
    print("Chapter:", section["chapter"])
    print("Section:", section["section"])
    print(
        "Pages:",
        section["page_start"],
        "-",
        section["page_end"],
    )
    print("Text:", section["text"][:250])

Sections detected: 80

Section 1
Chapter: General
Section: Introduction
Pages: 1 - 1
Text: Model: YY2984

Section 2
Chapter: General
Section: Download the app, and set up the headset
Pages: 1 - 1
Text: Many useful features are out there in your device.
Download the “Sony | Sound Connect” and get the most out of what you have.
Let’s start to use

Section 3
Chapter: General
Section: Turning on the headset
Pages: 1 - 1
Text: Press and hold the (power) button for about 2 seconds or more until the indicator (blue) flashes.
The headset enters pairing mode automatically when you turn on the headset for the first time after
purchase.

Section 4
Chapter: General
Section: Establishing a BLUETOOTH® connection
Pages: 2 - 2
Text: Download the “Sony | Sound Connect” app.
Follow the downloaded “Sony | Sound Connect” app instructions.

Section 5
Chapter: General
Section: Operating the headset
Pages: 2 - 2
Text: You can use the touch sensor control panel to perform the following operations.
Tap twice q

# **Chunking**

In [ ]:
from chunker import build_chunks

chunks = build_chunks(sections)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5], start=1):
    print(f"\nChunk {i}")
    print("Chapter:", chunk["chapter"])
    print("Section:", chunk["section"])
    print(
        "Pages:",
        chunk["page_start"],
        "-",
        chunk["page_end"],
    )
    print("Characters:", len(chunk["text"]))
    print("Text:", chunk["text"][:350])

Total chunks: 191

Chunk 1
Chapter: General
Section: Introduction
Pages: 1 - 1
Characters: 13
Text: Model: YY2984

Chunk 2
Chapter: General
Section: Download the app, and set up the headset
Pages: 1 - 1
Characters: 144
Text: Many useful features are out there in your device.
Download the “Sony | Sound Connect” and get the most out of what you have.
Let’s start to use

Chunk 3
Chapter: General
Section: Turning on the headset
Pages: 1 - 1
Characters: 207
Text: Press and hold the (power) button for about 2 seconds or more until the indicator (blue) flashes.
The headset enters pairing mode automatically when you turn on the headset for the first time after
purchase.

Chunk 4
Chapter: General
Section: Establishing a BLUETOOTH® connection
Pages: 2 - 2
Characters: 103
Text: Download the “Sony | Sound Connect” app.
Follow the downloaded “Sony | Sound Connect” app instructions.

Chunk 5
Chapter: General
Section: Operating the headset
Pages: 2 - 2
Characters: 466
Text: You can use the touch sens

# **Build the Chroma index**

In [ ]:
index_summary = pipeline.build_index(str(pdf_path))

index_summary

{'pages': 143,
 'sections': 80,
 'chunks': 191,
 'section_preview': [{'chapter': 'General',
   'section': 'Introduction',
   'text': 'Model: YY2984',
   'page_start': 1,
   'page_end': 1,
   'lines': [{'text': 'Model: YY2984', 'page': 1}]},
  {'chapter': 'General',
   'section': 'Download the app, and set up the headset',
   'text': 'Many useful features are out there in your device.\nDownload the “Sony | Sound Connect” and get the most out of what you have.\nLet’s start to use',
   'page_start': 1,
   'page_end': 1,
   'lines': [{'text': 'Many useful features are out there in your device.',
     'page': 1},
    {'text': 'Download the “Sony | Sound Connect” and get the most out of what you have.',
     'page': 1},
    {'text': 'Let’s start to use', 'page': 1}]},
  {'chapter': 'General',
   'section': 'Turning on the headset',
   'text': 'Press and hold the (power) button for about 2 seconds or more until the indicator (blue) flashes.\nThe headset enters pairing mode automatically when 

# **Semantic retrieval**

In [ ]:
question = "How do I reset my headphones?"

retrieved = pipeline.retrieve(
    question,
    top_k=TOP_K,
)

for rank, item in enumerate(retrieved, start=1):
    meta = item["metadata"]

    print(f"\n--- Result {rank} ---")
    print("Distance:", round(item["distance"], 4))
    print("Chapter:", meta["chapter"])
    print("Section:", meta["section"])
    print(
        "PDF pages:",
        meta["page_start"],
        "-",
        meta["page_end"],
    )
    print(item["text"][:700])


--- Result 1 ---
Distance: 0.4569
Chapter: General
Section: Resetting the headset
PDF pages: 141 - 142
If the headset cannot be turned on or cannot be operated even when it is turned on, reset the headset.
1 Connect the headset to an AC outlet.
Use the supplied USB Type-C cable and a commercially available USB AC adaptor (A).
2 Press and hold the (power) button (B) and the NC/AMB (noise canceling/Ambient Sound Mode) button
(C) for about 5 seconds.
The headset will be reset (the charging indicator turns off and then lights up again).
The pairing information and other settings are retained.
If the headset does not operate correctly even after resetting, initialize the headset to restore factory settings.
Related Topic

--- Result 2 ---
Distance: 0.4936
Chapter: General
Section: Connecting to a paired computer (Windows 10)
PDF pages: 66 - 69
.
Check the connection status on the computer. If it is not connected, proceed to step 3.
3 On your computer screen, right-click the speaker icon on

# **Generate an answer**

In [ ]:
result = pipeline.ask(
    "How do I turn on noise cancelling?",
    top_k=TOP_K,
)

display(Markdown("### Question"))
display(Markdown(result["question"]))

display(Markdown("### Answer"))
display(Markdown(result["answer"]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Question

How do I turn on noise cancelling?

### Answer

To turn on the noise canceling function:

1. Turn on the headset.
2. You will hear a notification sound indicating that the headset is turned on.
3. Press the NC/AMB (noise canceling/Ambient Sound Mode) button (A) to switch to the Noise Canceling Mode.
4. Each time the button is pressed, the function switches as follows:
   - Noise canceling function: ON
   - You will hear a notification sound.
   - Ambient Sound Mode: ON
   - You will hear a notification sound.
   
This process allows you to enjoy music without being disturbed by ambient noise.

# **Try multiple questions**

In [ ]:
test_questions = [
    "How do I pair my headphones with Android?",
    "How do I reset my headphones?",
    "How do I turn on Noise Canceling?",
    "How many devices can be paired with the headphones?",
    "What is the battery life with LC3 and ANC enabled?",
]

for question in test_questions:
    result = pipeline.ask(question)

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("ANSWER:", result["answer"])

    print("\nRETRIEVED SOURCES:")

    for source in result["sources"]:
        meta = source["metadata"]
        print(
            f"- {meta['chapter']} / {meta['section']} "
            f"(pages {meta['page_start']}-{meta['page_end']})"
        )


QUESTION: How do I pair my headphones with Android?
ANSWER: To pair your Sony WH-1000XM6 headphones with an Android smartphone, follow these steps:

1. Ensure the Android smartphone is within 1 meter of the headset.
2. Make sure the headset is charged sufficiently.
3. Have the operating instructions for the Android smartphone handy.
4. Turn on the headset when pairing for the first time after purchase or initialization.
5. Press and hold the Bluetooth button (A) for about 5 seconds or more to enter pairing mode manually.
6. The indicator (blue) will repeatedly flash twice in a row, followed by a voice guidance saying "Pairing".
7. Unlock the screen of the Android smartphone if it is locked.
8. Go to Settings > Device connection > Bluetooth.
9. Turn on the Bluetooth function.
10. Touch WH-1000XM6 on the Android smartphone screen.
11. If a passkey input is required, enter "0000".
12. The headset and Android smartphone will be paired and connected once the connection is established.
13. 

# **Launch Gradio**

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
from google.colab import output

PROJECT_DIR = Path("/content/drive/MyDrive/sony_rag_assistant")
os.chdir(PROJECT_DIR)

log_path = "/content/streamlit.log"

# Allow requests forwarded through the Colab proxy.
env = os.environ.copy()
env["STREAMLIT_SERVER_ENABLE_CORS"] = "false"
env["STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION"] = "false"

log_file = open(log_path, "w")

process = subprocess.Popen(
    [
        sys.executable,
        "-m", "streamlit", "run",
        str(PROJECT_DIR / "app.py"),
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--browser.gatherUsageStats=false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)

time.sleep(5)

if process.poll() is not None:
    log_file.flush()
    print(Path(log_path).read_text()[-5000:])
else:
    print("Streamlit is running.")
    print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

Streamlit is running.
https://8501-gpu-t4-s-ass1b0-zbqdlm6sd9ij-b.asia-southeast1.prod.r.colab.dev
